# 🔍 Error Analysis — Email Writing Assistant

Qualitative analysis of model failures and edge cases.

This notebook helps identify:
- Common failure modes (repetition, off-topic, wrong tone)
- Worst-performing samples by metric
- Length distribution mismatches
- Task-specific weaknesses (compose vs reply)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from collections import Counter

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Load evaluation results
RESULTS_FILE = "../evaluation/results.json"  # Adjust path

with open(RESULTS_FILE, 'r') as f:
    results = json.load(f)

per_sample = results['fine_tuned']['per_sample']
df = pd.DataFrame(per_sample)

print(f"Total samples: {len(df)}")
print(f"Task distribution:\n{df['task_type'].value_counts()}")

## 1. Failure Mode Categorization

In [ ]:
def categorize_failure(row):
    """Classify the type of failure for a sample."""
    pred = row.get('prediction', '')
    ref = row.get('reference', '')
    rouge1 = row.get('rouge1', 0)
    
    failures = []
    
    # Check for repetition (same phrase repeated 3+ times)
    words = pred.lower().split()
    if len(words) > 10:
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        trigram_counts = Counter(trigrams)
        max_repeat = max(trigram_counts.values()) if trigram_counts else 0
        if max_repeat >= 3:
            failures.append('repetition')
    
    # Check for too short
    if len(pred.split()) < 10:
        failures.append('too_short')
    
    # Check for too long (2x reference length)
    if len(pred.split()) > 2 * len(ref.split()) and len(ref.split()) > 10:
        failures.append('too_long')
    
    # Check for missing greeting
    greeting_re = re.compile(r'^(dear|hi|hello|hey|good)', re.IGNORECASE | re.MULTILINE)
    if not greeting_re.search(pred):
        failures.append('no_greeting')
    
    # Check for missing closing
    closing_re = re.compile(r'(regards|sincerely|best|thanks|thank you)', re.IGNORECASE)
    if not closing_re.search(pred):
        failures.append('no_closing')
    
    # Check for very low ROUGE (off-topic)
    if rouge1 < 0.1:
        failures.append('off_topic')
    
    return failures if failures else ['none']

df['failures'] = df.apply(categorize_failure, axis=1)
df['failure_str'] = df['failures'].apply(lambda x: ', '.join(x))

# Count failure types
all_failures = [f for failures in df['failures'] for f in failures]
failure_counts = Counter(all_failures)

print("Failure Mode Distribution:")
for mode, count in failure_counts.most_common():
    print(f"  {mode:<20s} {count:>5d} ({100*count/len(df):.1f}%)")

In [ ]:
# Visualize failure mode distribution
fig, ax = plt.subplots(figsize=(10, 5))

modes = [m for m, _ in failure_counts.most_common()]
counts = [c for _, c in failure_counts.most_common()]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(modes)))

bars = ax.barh(modes, counts, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Count')
ax.set_title('Failure Mode Distribution', fontsize=16, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for bar in bars:
    w = bar.get_width()
    ax.annotate(f'{w}', xy=(w, bar.get_y() + bar.get_height()/2),
                xytext=(5, 0), textcoords='offset points', ha='left', va='center')

plt.tight_layout()
plt.show()

## 2. Worst-Performing Samples

In [ ]:
# Show the 5 worst samples by ROUGE-1
worst_samples = df.nsmallest(5, 'rouge1')

for _, row in worst_samples.iterrows():
    print(f"\n{'='*60}")
    print(f"Task: {row['task_type']} | ROUGE-1: {row['rouge1']:.4f} | Failures: {row['failure_str']}")
    print(f"{'='*60}")
    print(f"\n📝 Prompt: {row['prompt'][:200]}")
    print(f"\n🤖 Generated:\n{row['prediction'][:300]}")
    print(f"\n✅ Reference:\n{row['reference'][:300]}")

## 3. Per-Task Analysis

In [ ]:
# Compare metrics by task type
metric_cols = ['rouge1', 'rouge2', 'rougeL', 'bleu', 'format_compliance']
available_cols = [c for c in metric_cols if c in df.columns]

if available_cols:
    task_summary = df.groupby('task_type')[available_cols].agg(['mean', 'std'])
    print("\nMetrics by Task Type:")
    print(task_summary.round(4).to_string())

In [ ]:
# Box plot of ROUGE-1 by task type
if 'rouge1' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    task_types = df['task_type'].unique()
    data_by_task = [df[df['task_type'] == t]['rouge1'].values for t in task_types]
    
    bp = ax.boxplot(data_by_task, labels=task_types, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    
    colors = ['#818cf8', '#34d399']
    for patch, color in zip(bp['boxes'], colors[:len(task_types)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_ylabel('ROUGE-1')
    ax.set_title('ROUGE-1 Distribution by Task Type', fontsize=16, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 4. Length Distribution Analysis

In [ ]:
# Compare word count distributions
df['pred_word_count'] = df['prediction'].apply(lambda x: len(str(x).split()))
df['ref_word_count'] = df['reference'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['pred_word_count'], bins=30, alpha=0.7, color='#818cf8', label='Generated', edgecolor='white', linewidth=0.5)
axes[0].hist(df['ref_word_count'], bins=30, alpha=0.5, color='#34d399', label='Reference', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Scatter: generated vs reference length
axes[1].scatter(df['ref_word_count'], df['pred_word_count'], alpha=0.5, color='#818cf8', s=20)
max_val = max(df['ref_word_count'].max(), df['pred_word_count'].max())
axes[1].plot([0, max_val], [0, max_val], '--', color='#94a3b8', linewidth=1, label='y=x')
axes[1].set_xlabel('Reference Word Count')
axes[1].set_ylabel('Generated Word Count')
axes[1].set_title('Length Correlation', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nGenerated: mean={df['pred_word_count'].mean():.1f} ± {df['pred_word_count'].std():.1f} words")
print(f"Reference: mean={df['ref_word_count'].mean():.1f} ± {df['ref_word_count'].std():.1f} words")

## 5. Repetition Analysis

In [ ]:
# Detect and quantify repetition
def repetition_score(text):
    """Ratio of repeated trigrams to total trigrams (higher = more repetitive)."""
    words = text.lower().split()
    if len(words) < 4:
        return 0.0
    trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    counts = Counter(trigrams)
    repeated = sum(c - 1 for c in counts.values() if c > 1)
    return repeated / len(trigrams) if trigrams else 0.0

df['repetition_score'] = df['prediction'].apply(repetition_score)

print(f"Mean repetition score: {df['repetition_score'].mean():.4f}")
print(f"Max repetition score: {df['repetition_score'].max():.4f}")
print(f"Samples with high repetition (>0.1): {(df['repetition_score'] > 0.1).sum()}")

# Show most repetitive samples
if (df['repetition_score'] > 0.1).any():
    print("\n--- Most Repetitive Samples ---")
    for _, row in df.nlargest(3, 'repetition_score').iterrows():
        print(f"\nRepetition: {row['repetition_score']:.3f} | Task: {row['task_type']}")
        print(f"Generated: {row['prediction'][:300]}...")

## 6. Summary & Recommendations

In [ ]:
print("="*60)
print("ERROR ANALYSIS SUMMARY")
print("="*60)
print(f"\nTotal samples analyzed: {len(df)}")
print(f"\nFailure breakdown:")
for mode, count in failure_counts.most_common():
    if mode != 'none':
        print(f"  • {mode}: {count} ({100*count/len(df):.1f}%)")

no_issue_count = failure_counts.get('none', 0)
print(f"\n✅ Clean samples (no issues): {no_issue_count} ({100*no_issue_count/len(df):.1f}%)")

print(f"\nMean ROUGE-1: {df['rouge1'].mean():.4f}")
print(f"Mean repetition score: {df['repetition_score'].mean():.4f}")

print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)
if failure_counts.get('repetition', 0) > len(df) * 0.05:
    print("⚠️  High repetition rate — consider increasing repetition penalty during generation")
if failure_counts.get('no_greeting', 0) > len(df) * 0.3:
    print("⚠️  Many emails lack greetings — consider adjusting system prompt")
if failure_counts.get('no_closing', 0) > len(df) * 0.3:
    print("⚠️  Many emails lack closings — consider adding closing examples to training data")
if failure_counts.get('too_short', 0) > len(df) * 0.1:
    print("⚠️  Many short responses — consider increasing min_new_tokens")
print("\n✅ Analysis complete.")